# 16.18 - YOLO11s matched capacity comparison

This notebook trains a source-faithful two-class detector on the authorized Roboflow dataset subset. The retained source labels are `gap_or_spacing` and `malposition_present`; do not describe them as precise clinical diagnoses of midline diastema or orthodontic crowding. The experiment uses the locked source train/validation split, never the held-out test set.

**Research role:** YOLO11s capacity comparison with COCO initialization. Source data and annotation limitations are summarized in the attached dataset card.

## 1. Research motivation and paper anchor

The 2026 YOLOv11 malocclusion study uses bounding-box detection on standardized intraoral views. It reports strong frontal diastema and occlusal crowding results but explicitly states that its institutional images are not publicly available. We use its view-specific reasoning and YOLO detector family as method context, not as a reproduction of its dataset or metrics.

The current public source has different, broader classes and less standardized phone-style photos. The current experiment asks whether a compact YOLO11 detector can learn those source labels under a reproducible split.

Paper: Nebiaj et al. (2026), *Multi-Class Malocclusion Detection on Standardized Intraoral Photographs Using YOLOv11*.

In [ ]:
PAPER_URL = "https://www.mdpi.com/2304-6767/14/1/60"
DATASET_URL = "https://universe.roboflow.com/okasha/dental-ktzqf"
DATASET_ARCHIVE_SHA256 = "32c344cf993b9d4c48d452d78de63359325a2cccfeba6f04bb565c1052ff5f25"
MODEL_LABEL = "YOLO11s capacity comparison with COCO initialization"
print("Paper:", PAPER_URL)
print("Dataset:", DATASET_URL)
print("Experiment:", MODEL_LABEL)

## 2. Install and pin the training library

A fixed Ultralytics version makes the model implementation and training options reproducible across the parallel Kaggle runs.

In [ ]:
%pip install -q ultralytics==8.3.228

## 3. Attach and verify the prepared data

The Kaggle input contains one ZIP with only train and validation images, the hashed manifest, the two-class label map, and source attribution. The ZIP SHA-256 binds the run to the exact prepared package. Extraction checks for unsafe paths and confirms that no test directory exists.

The cell verifies the full manifest checksum and split counts, then hashes eight fixed-seed image/label pairs per split to keep Kaggle startup bounded.


In [ ]:
from pathlib import Path
from hashlib import sha256
import json, shutil, time, yaml, torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display
from ultralytics import YOLO

EXPECTED_MANIFEST_SHA256 = "38cba07194707131b629b020ea9078d487dd397e13221d88075ca8d9cf7a3e9b"
manifest_candidates = list(Path("/kaggle/input").rglob("manifest.csv"))
assert len(manifest_candidates) == 1, f"Expected one attached dataset manifest, found {manifest_candidates}"
manifest_path = manifest_candidates[0]
DATA_ROOT = manifest_path.parent
actual_manifest_sha = sha256(manifest_path.read_bytes()).hexdigest()
assert actual_manifest_sha == EXPECTED_MANIFEST_SHA256, (actual_manifest_sha, EXPECTED_MANIFEST_SHA256)
manifest = pd.read_csv(manifest_path, dtype={"sample_id": str, "group_id_sha256_20": str})
assert len(manifest) == 15511 and manifest.sample_id.is_unique
assert set(manifest.split) == {"train", "valid"}
assert not (set(manifest.loc[manifest.split=="train", "group_id_sha256_20"]) & set(manifest.loc[manifest.split=="valid", "group_id_sha256_20"]))
SOURCE_DATA_YAML = DATA_ROOT / "data.yaml"
config = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding="utf-8"))
config["path"] = DATA_ROOT.as_posix()
config["train"] = "train/images"
config["val"] = "valid/images"
config["names"] = {0: "gap_or_spacing", 1: "malposition_present"}
DATA_YAML = Path("/kaggle/working/alignment_visible_v2_data.yaml")
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

# Fast integrity gate: verify expected split inventory, then hash a fixed seeded sample of image-label pairs.
expected_counts = {"train": 15034, "valid": 477}
for split, expected_count in expected_counts.items():
    image_count = len(list((DATA_ROOT / split / "images").glob("*")))
    label_count = len(list((DATA_ROOT / split / "labels").glob("*")))
    assert image_count == label_count == expected_count, (split, image_count, label_count)
sample_rows = manifest.groupby("split", group_keys=False).sample(n=8, random_state=51324)
for row in sample_rows.itertuples(index=False):
    for rel_path, expected_hash in ((row.image, row.image_sha256), (row.label, row.label_sha256)):
        file_path = DATA_ROOT / rel_path
        assert file_path.is_file(), f"Missing sampled manifest file: {rel_path}"
        digest = sha256(file_path.read_bytes()).hexdigest()
        assert digest == expected_hash, f"Sample content checksum mismatch: {rel_path}"
print("Verified Kaggle dataset manifest SHA-256:", actual_manifest_sha)
print("Verified sampled image/label hashes:", 2 * len(sample_rows))
print("Train/validation image and label counts:", expected_counts)
print("Filename-derived source groups:", manifest.groupby("split").group_id_sha256_20.nunique().to_dict())
print("Output classes:", config["names"])
display(manifest.groupby("split")[["gap_or_spacing_boxes", "malposition_boxes"]].sum())

## 4. Review the annotation geometry and one training example

This cell confirms that the two-class YOLO text files contain valid normalized boxes and draws one source example. The visualization is for annotation-format review only; it is not a diagnostic result.

In [ ]:
def read_yolo(label_path):
    rows = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        cls, x, y, w, h = line.split()
        cls = int(cls); x, y, w, h = map(float, (x, y, w, h))
        assert cls in (0, 1)
        assert 0 <= x-w/2 < x+w/2 <= 1 and 0 <= y-h/2 < y+h/2 <= 1
        rows.append((cls, x, y, w, h))
    return rows

positive_train = manifest[(manifest.split == "train") & (manifest.target_boxes > 0)]
row = positive_train.iloc[0]
image_path = DATA_ROOT / row.image
label_path = DATA_ROOT / row.label
image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
for cls, x, y, w, h in read_yolo(label_path):
    color = "#00a000" if cls == 0 else "#d00000"
    draw.rectangle(((x-w/2)*image.width, (y-h/2)*image.height, (x+w/2)*image.width, (y+h/2)*image.height), outline=color, width=3)
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.title(f"Source labels: {row.gap_or_spacing_boxes} spacing, {row.malposition_boxes} malposition"); plt.axis("off"); plt.show()
print("Example sample ID:", row.sample_id)
print("Target boxes:", len(read_yolo(label_path)))

## 5. Confirm the Kaggle GPU and training setup

The run uses two T4 GPUs when available, a fixed seed, AdamW, mild augmentation, early stopping, periodic checkpoints, and a time budget below the Kaggle session limit. Baseline comparisons share the same data and recipe; only model initialization/capacity changes.

In [ ]:
DEVICE = "0,1" if torch.cuda.device_count() >= 2 else "0"
assert torch.cuda.is_available()
print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
model = YOLO("yolo11s.pt")
print("Initialization:", "COCO-pretrained yolo11s.pt")

## 6. Train with a bounded, resumable configuration

The 50-epoch cap and 1.75-hour training budget leave room for Kaggle startup and output collection. Ultralytics writes best/last checkpoints and results during training; periodic checkpoints are saved every five epochs. Treat a budget-stopped run as partial and resume only with the same data and protocol.

In [ ]:
import json, time
from pathlib import Path

TRAIN_CONFIG = {
    "model": MODEL_LABEL,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "seed": 51324,
    "epochs": 50,
    "patience": 8,
    "imgsz": 640,
    "batch": 32,
    "device": DEVICE,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "workers": 4,
    "source_split": "locked source train/valid; source test excluded",
}
print(json.dumps(TRAIN_CONFIG, indent=2))

started = time.monotonic()
result = model.train(
    data=str(DATA_YAML),
    epochs=TRAIN_CONFIG["epochs"],
    patience=TRAIN_CONFIG["patience"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    device=TRAIN_CONFIG["device"],
    workers=TRAIN_CONFIG["workers"],
    seed=TRAIN_CONFIG["seed"],
    deterministic=True,
    optimizer=TRAIN_CONFIG["optimizer"],
    lr0=TRAIN_CONFIG["lr0"],
    cos_lr=True,
    amp=True,
    fliplr=0.5, flipud=0.0, degrees=5.0, translate=0.05, scale=0.10,
    hsv_h=0.005, hsv_s=0.25, hsv_v=0.15, mosaic=0.5, close_mosaic=8,
    save=True, save_period=5, plots=True, time=1.75,
    project="/kaggle/working/runs", name="16_18_visible_yolo11s_comparison", exist_ok=True,
)
run_dir = Path("/kaggle/working/runs") / TRAIN_CONFIG["run_name"]
assert run_dir.is_dir() and (run_dir / "weights" / "best.pt").is_file(), "Training ended without a saved best checkpoint"
print("Training elapsed seconds:", round(time.monotonic()-started, 1))
print("Run directory:", run_dir)
print("Training result rows:", len(pd.read_csv(run_dir / "results.csv")))
(run_dir / "run_config.json").write_text(json.dumps(TRAIN_CONFIG | {"run_directory": str(run_dir)}, indent=2), encoding="utf-8")

## 7. Validate the selected checkpoint and save evidence

Validation uses only the locked source validation set. Report aggregate mAP50 and mAP50-95, per-class AP, the confusion matrix, runtime, and checkpoint hash. These values are exploratory because the source has no patient IDs and the labels are broad visual categories.

In [ ]:
import hashlib
best_path = run_dir / "weights" / "best.pt"
assert best_path.is_file(), "No best checkpoint was produced"
final_model = YOLO(str(best_path))
val_metrics = final_model.val(data=str(DATA_YAML), imgsz=640, batch=32, device=DEVICE, plots=True, project=str(run_dir), name="final_validation", exist_ok=True)
metric_report = {"aggregate": val_metrics.results_dict, "checkpoint_sha256": hashlib.sha256(best_path.read_bytes()).hexdigest(), "checkpoint_bytes": best_path.stat().st_size, "manifest_sha256": EXPECTED_MANIFEST_SHA256, "class_names": config["names"], "run_config": TRAIN_CONFIG}
(run_dir / "final_validation_metrics.json").write_text(json.dumps(metric_report, indent=2, default=float), encoding="utf-8")
print(json.dumps(metric_report, indent=2, default=float))
if hasattr(val_metrics, "box"):
    display(pd.DataFrame({"class_id": val_metrics.box.ap_class_index, "AP50": val_metrics.box.ap50, "AP50_95": val_metrics.box.ap}))
print("Saved evidence:", run_dir)

## 8. Interpretation boundary

A high validation score is not a clinical result, a phone-generalization guarantee, or proof that the broad source labels equal paper-defined diastema/crowding. The source includes multiple camera views, pre-generated train augmentations, no verified patient groups, and an unresolved version-metadata inconsistency. Compare this run only with the matched notebooks using the same data version, split, seed, resolution, and augmentation recipe.